# Ejercicio Módulo 2
**Inteligencia Artificial - CEIA - FIUBA**

**Tomas Moretti**

En este ejercicio deben implementar un algoritmo de búsqueda que no sea **Búsqueda Primero en Anchura (BFS)** para resolver el problema de la Torre de Hanoi. La nota máxima dependerá del algoritmo implementado:

- **Búsqueda Primero en Profundidad**: nota máxima 6.
- **Búsqueda de Costo Uniforme**: nota máxima 6.
- **Búsqueda de Profundidad Limitada con Profundidad Iterativa**: nota máxima 7.
- **Búsqueda Voraz usando la heurística dada en el aula virtual**: nota máxima 8.
- **Búsqueda Voraz usando una heurística desarrollada por vos**: nota máxima 9.
- **Búsqueda A\* usando la heurística dada en el aula virtual**: nota máxima 9.
- **Búsqueda A\* usando una heurística desarrollada por vos**: nota máxima 10.

La función debe devolver la salida correspondiente a la solución encontrada o `None si no se encontró una solución.

Además, debe calcular métricas de rendimiento que, como mínimo, incluyan:

- `solution_found`: `True` si se encontró la solución, `False` en caso contrario.
- `nodes_explored`: cantidad de nodos explorados (entero).
- `states_visited`: cantidad de estados distintos visitados (entero).
- `nodes_in_frontier`: cantidad de nodos que quedaron en la frontera al finalizar la ejecución (entero).
- `max_depth`: máxima profundidad explorada (entero).
- `cost_total`: costo total para encontrar la solución (float).

In [7]:
from aima_libs.aima import PriorityQueue
from aima_libs.hanoi_states import ProblemHanoi, StatesHanoi
from aima_libs.tree_hanoi import NodeHanoi

In [8]:
def search_algorithm(number_disks=5) -> tuple[NodeHanoi, dict]:

    list_disks = [i for i in range(number_disks, 0, -1)]
    initial_state = StatesHanoi(list_disks, [], [], max_disks=number_disks)
    goal_state = StatesHanoi([], [], list_disks, max_disks=number_disks)
    problem = ProblemHanoi(initial=initial_state, goal=goal_state)

    ##### EDITAR ESTA ZONA
    from aima_libs.aima import PriorityQueue

    # Varilla destino: la única que tiene discos en el estado objetivo.
    goal_rod = next(i for i, rod in enumerate(goal_state.rods) if rod)

    def heuristic(state: StatesHanoi) -> float:
        """h(n) = 2^(d-1), donde d es el disco mal ubicado más grande.

        Para colocar el disco d en su lugar hay que sacarle de encima los d-1
        discos más chicos y amontonarlos en la varilla auxiliar, lo que cuesta
        como mínimo 2^(d-1) - 1 movimientos, más 1 para mover el disco d.
        Como cada término es un mínimo inevitable, nunca sobreestima.
        """
        # Un disco está bien ubicado si él y todos los más grandes ya están en
        # la varilla destino. Como los bien ubicados son siempre los más
        # grandes y van seguidos, su cantidad determina cuál es el disco mal
        # ubicado más grande.
        bien_ubicados = 0
        for posicion, disco in enumerate(state.rods[goal_rod]):
            if disco == number_disks - posicion:
                bien_ubicados += 1
            else:
                break

        disco_mayor_mal_ubicado = number_disks - bien_ubicados
        if disco_mayor_mal_ubicado == 0:
            return 0.0

        return float(2 ** (disco_mayor_mal_ubicado - 1))

    def f(node: NodeHanoi) -> float:
        """f(n) = g(n) + h(n)."""
        return node.path_cost + heuristic(node.state)

    # Frontera: cola de prioridad ordenada por f(n), menor primero.
    root = NodeHanoi(problem.initial)
    frontier = PriorityQueue(order="min", f=f)
    frontier.append(root)

    best_cost = {root.state: root.path_cost}  # mejor g conocido por estado
    explored = set()                          # estados ya expandidos

    solution = None
    nodes_explored = 0
    max_depth = 0

    while len(frontier) > 0:
        _, node = frontier.pop()  # pop() devuelve la tupla (f, nodo)

        if node.state in explored:
            continue

        explored.add(node.state)
        nodes_explored += 1
        max_depth = max(max_depth, node.depth)

        if problem.goal_test(node.state):
            solution = node
            break

        for child in node.expand(problem):
            if child.state in explored:
                continue
            previous = best_cost.get(child.state)
            if previous is None or child.path_cost < previous:
                best_cost[child.state] = child.path_cost
                frontier.append(child)

    metrics = {
        "solution_found": solution is not None,
        "nodes_explored": nodes_explored,
        "states_visited": len(explored),
        "nodes_in_frontier": len(frontier),
        "max_depth": max_depth,
        "cost_total": float(solution.state.accumulated_cost) if solution else None,
    }

    #####

    return solution, metrics

Se prueba la función:

In [9]:
solution, metrics = search_algorithm(number_disks=5)

Veamos las métricas:

In [10]:
for key, value in metrics.items():
    print(f"{key}: {value}")

solution_found: True
nodes_explored: 122
states_visited: 122
nodes_in_frontier: 5
max_depth: 31
cost_total: 31.0


Veamos el camino de estados desde el principio a la solución:

In [5]:
for nodos in solution.path():
    print(nodos)

<Node HanoiState: 5 4 3 2 1 |  | >
<Node HanoiState: 5 4 3 2 |  | 1>
<Node HanoiState: 5 4 3 | 2 | 1>
<Node HanoiState: 5 4 3 | 2 1 | >
<Node HanoiState: 5 4 | 2 1 | 3>
<Node HanoiState: 5 4 1 | 2 | 3>
<Node HanoiState: 5 4 1 |  | 3 2>
<Node HanoiState: 5 4 |  | 3 2 1>
<Node HanoiState: 5 | 4 | 3 2 1>
<Node HanoiState: 5 | 4 1 | 3 2>
<Node HanoiState: 5 2 | 4 1 | 3>
<Node HanoiState: 5 2 1 | 4 | 3>
<Node HanoiState: 5 2 1 | 4 3 | >
<Node HanoiState: 5 2 | 4 3 | 1>
<Node HanoiState: 5 | 4 3 2 | 1>
<Node HanoiState: 5 | 4 3 2 1 | >
<Node HanoiState:  | 4 3 2 1 | 5>
<Node HanoiState: 1 | 4 3 2 | 5>
<Node HanoiState: 1 | 4 3 | 5 2>
<Node HanoiState:  | 4 3 | 5 2 1>
<Node HanoiState: 3 | 4 | 5 2 1>
<Node HanoiState: 3 | 4 1 | 5 2>
<Node HanoiState: 3 2 | 4 1 | 5>
<Node HanoiState: 3 2 1 | 4 | 5>
<Node HanoiState: 3 2 1 |  | 5 4>
<Node HanoiState: 3 2 |  | 5 4 1>
<Node HanoiState: 3 | 2 | 5 4 1>
<Node HanoiState: 3 | 2 1 | 5 4>
<Node HanoiState:  | 2 1 | 5 4 3>
<Node HanoiState: 1 | 2 | 5 4 

Y las acciones que el agente debería aplicar para llegar al objetivo:

In [6]:
for act in solution.solution():
    print(act)

Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 4 from 1 to 2
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 1
Move disk 3 from 3 to 2
Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 5 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 3 from 2 to 1
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 1
Move disk 4 from 2 to 3
Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
